# 0.0 Graph Sanity Check

**Learning:**
- [0.0 Getting Started](../../../Learning/LangGraph/00_foundations/0.0_getting_started.md)
- [1.1 Mental Model](../../../Learning/LangGraph/00_foundations/1.1_mental_model.md)

**Goal:** Confirm LangGraph is installed, build your first `StateGraph`, and compare it side-by-side with an LCEL pipeline.

## Setup

In [3]:
import sys
from pathlib import Path

# Find repo root (directory that contains src/ and requirements.txt)
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

# Returns True if .env file was found; False if not (Docker may already inject vars via env_file)
loaded = load_dotenv(ROOT / ".env")
print(f"Project root: {ROOT}")
print(f"load_dotenv: {loaded}  (.env path: {ROOT / '.env'})")

Project root: /app
load_dotenv: True  (.env path: /app/.env)


## 1. Verify LangGraph Installation

In [4]:
from importlib.metadata import version, PackageNotFoundError
from langgraph.graph import StateGraph, START, END

try:
    print(f"LangGraph version: {version('langgraph')}")
except PackageNotFoundError:
    print("LangGraph version: (installed, version metadata unavailable)")

print("Imports OK: StateGraph, START, END")

LangGraph version: 0.2.53
Imports OK: StateGraph, START, END


## 2. Your First StateGraph

A graph has three parts:
1. **State** — typed data shared between nodes
2. **Nodes** — functions that read and update state
3. **Edges** — connections that define execution order

Flow: `START → greet → END`

In [5]:
from typing import TypedDict  # TypedDict = dict with named, typed fields for graph state


class State(TypedDict):
    message: str  # the only field in our graph state: a string message


def greet(state: State) -> State:
    # Node function: receives full state, returns a partial update
    return {"message": f"Hello, {state['message']}!"}
    # Read input message, prepend "Hello, ", write back to state


graph = StateGraph(State)
# Create an empty graph builder; all nodes share the State schema

graph.add_node("greet", greet)
# Register a node named "greet" that runs the greet() function

graph.add_edge(START, "greet")
# When the graph starts, always go to the "greet" node first

graph.add_edge("greet", END)
# After "greet" finishes, stop the graph (no more nodes)

app = graph.compile()
# Freeze the graph into a runnable application

result = app.invoke({"message": "LangGraph"})
# Run once with initial state; graph executes START → greet → END

result
# Display final state: {'message': 'Hello, LangGraph!'}

{'message': 'Hello, LangGraph!'}

## 3. LCEL vs. StateGraph (Same Transform, Different Model)

| LCEL | LangGraph |
|---|---|
| Linear pipeline | Explicit nodes and edges |
| Great for stateless transforms | Great when steps branch, loop, or persist |

Both approaches below apply the same greeting transform.

In [6]:
from langchain_core.runnables import RunnableLambda

# LCEL: pipe-style transformation
lcel_chain = RunnableLambda(lambda x: {"message": f"Hello, {x['message']}!"})
lcel_result = lcel_chain.invoke({"message": "LangGraph"})

print("LCEL result:", lcel_result)
print("Graph result:", result)
print("Same output:", lcel_result == result)

LCEL result: {'message': 'Hello, LangGraph!'}
Graph result: {'message': 'Hello, LangGraph!'}
Same output: True


## 4. Optional — LCEL with LLM vs. Graph with LLM Node

Skip this cell if `OPENAI_API_KEY` is not configured. It shows how an LLM fits **inside** a graph node—the pattern used throughout this track.

### Concept Note — read before running Part 4

Part 4 is a **comparison demo**, not a production pattern.

| Layer | Question it answers |
|---|---|
| **LCEL** (`prompt \| llm \| parser`) | How does **one step** work? |
| **LangGraph** (`StateGraph`) | What runs **next**, in what **order**, with what **shared state**? |

In Part 4 you will:
1. Build an LCEL chain (Part A)
2. Call the **same chain** inside a graph node (Part B)
3. Compare outputs to prove they produce the same result

**Why reuse `lcel_llm_chain` inside `llm_node`?**  
So any difference in output would come from LangGraph—not from changed logic. For a single-node graph (`START → llm → END`), plain LCEL is enough. LangGraph earns its value once you add branching, multiple nodes, loops, or memory (see Section 5 below).

**What you register in the graph:** the **node function** (`llm_node`), not the LCEL chain itself.

```python
graph.add_node("llm", llm_node)   # ✅ register the function
# graph.add_node("llm", lcel_llm_chain)  # ❌ not how it works
```

In [7]:
# --- Imports ---
from langchain_core.prompts import ChatPromptTemplate   # reusable prompt template with {variables}
from langchain_core.output_parsers import StrOutputParser  # converts LLM response object → plain string
from src.config import OPENAI_API_KEY                   # reads OPENAI_API_KEY from .env / Docker env
from src.llms.openai_chat import make_openai_chat       # project helper that returns a configured ChatOpenAI

# --- Guard: skip if no API key ---
if not OPENAI_API_KEY:
    # .env missing or empty — avoid a runtime auth error
    print("Skipping LLM demo — set OPENAI_API_KEY in .env to run this cell.")
else:
    # --- Part A: LCEL pipeline (linear) ---

    llm = make_openai_chat()
    # Creates ChatOpenAI(model=..., temperature=0) using credentials from .env

    prompt = ChatPromptTemplate.from_template(
        "Say hello to {name} in one short sentence."
    )
    # Template with one variable `{name}` — filled at invoke time

    lcel_llm_chain = prompt | llm | StrOutputParser()
    # LCEL pipe:  input dict → prompt formats text → llm generates → parser extracts string
    # Equivalent flow: {"name": "..."} → "Say hello to ..." → AIMessage → "Hello, ..."

    lcel_reply = lcel_llm_chain.invoke({"name": "LangGraph"})
    # Run the chain once; pass the variable value for `{name}`

    print("LCEL + LLM:", lcel_reply)
    # Print the final plain-text string from the LCEL pipeline

    # --- Part B: LangGraph (same LLM logic wrapped as a node) ---

    class LlmState(TypedDict):
        name: str    # input: who to greet
        reply: str   # output: LLM response stored back into graph state

    def llm_node(state: LlmState) -> LlmState:
        # Node function: reads current state, returns a partial state update
        reply = lcel_llm_chain.invoke({"name": state["name"]})
        # Reuse the same LCEL chain inside the node
        return {"reply": reply}
        # LangGraph merges {"reply": ...} into the existing state (keeps `name`)

    llm_graph = StateGraph(LlmState)
    # Create a graph whose shared data shape is LlmState

    llm_graph.add_node("llm", llm_node)
    # Register one node named "llm" that runs llm_node()

    llm_graph.add_edge(START, "llm")
    # Entry point: when graph runs, go to the "llm" node first

    llm_graph.add_edge("llm", END)
    # After "llm" finishes, stop the graph

    llm_app = llm_graph.compile()
    # Turn the graph definition into a runnable app (like chain.compile())

    graph_reply = llm_app.invoke({"name": "LangGraph", "reply": ""})
    # Run graph once with initial state; `reply` starts empty and gets filled by the node

    print("Graph + LLM:", graph_reply["reply"])
    # Print the reply from final graph state — should match the LCEL result above

LCEL + LLM: Hello, LangGraph! Excited to connect and explore together!
Graph + LLM: Hello, LangGraph! Excited to connect and explore together!


### After Part 4 — did it feel redundant?

If Part B felt like unnecessary wrapping, that is **expected**.

```
Part A (LCEL):   input → prompt → llm → parser → output
Part B (Graph):  input → [one node that calls the same chain] → output
```

For one step, the graph adds little. The point is: **LangGraph nodes can call LCEL chains** — they are complementary layers.

**Common production pattern:**
1. Build LCEL pipeline for one step
2. Wrap it in a node function
3. Register the **function** in the graph

Not every node needs LCEL — your `greet` node (Section 2) was plain Python, and that is perfectly valid.

## 5. When LangGraph Adds Value — Multi-Node Scenario

The examples below use **plain Python only** (no API key) to show where graphs shine: **routing and multi-step workflows**.

**Scenario A — Support ticket router**

```
START → classify → [billing | technical | general] → END
```

LCEL alone is linear (`A | B | C`). It cannot easily express "go to different nodes based on input" without extra code. LangGraph makes routing explicit.

In [ ]:
# --- Scenario A: Support ticket router (no LLM, no LCEL) ---
# Demonstrates conditional routing — the main reason to use LangGraph over a single chain

class TicketState(TypedDict):
    message: str     # user's ticket text
    category: str    # filled by classify node
    response: str    # filled by the specialist node


def classify_node(state: TicketState) -> TicketState:
    # Node 1: plain Python logic — no LCEL needed
    msg = state["message"].lower()
    if "bill" in msg or "payment" in msg:
        category = "billing"
    elif "error" in msg or "bug" in msg:
        category = "technical"
    else:
        category = "general"
    return {"category": category}


def billing_node(state: TicketState) -> TicketState:
    return {"response": "Billing: we'll review your invoice within 24 hours."}


def technical_node(state: TicketState) -> TicketState:
    return {"response": "Tech support: please share your error logs and steps to reproduce."}


def general_node(state: TicketState) -> TicketState:
    return {"response": "Support: thanks for reaching out — an agent will follow up shortly."}


def route_ticket(state: TicketState) -> str:
    # Router function: returns the NAME of the next node (must match add_node names)
    return state["category"]


ticket_graph = StateGraph(TicketState)
ticket_graph.add_node("classify", classify_node)
ticket_graph.add_node("billing", billing_node)
ticket_graph.add_node("technical", technical_node)
ticket_graph.add_node("general", general_node)

ticket_graph.add_edge(START, "classify")
# Conditional edge: after "classify", go to different nodes based on route_ticket()
ticket_graph.add_conditional_edges("classify", route_ticket)
ticket_graph.add_edge("billing", END)
ticket_graph.add_edge("technical", END)
ticket_graph.add_edge("general", END)

ticket_app = ticket_graph.compile()

# Run two different inputs — watch the graph take different paths
for msg in ["I have a billing issue with my invoice", "The app throws an error on login"]:
    out = ticket_app.invoke({"message": msg, "category": "", "response": ""})
    print(f"Input:    {msg}")
    print(f"  Route:  {out['category']} → {out['response']}\n")

**Scenario B — Mixed nodes: Python + LCEL in one pipeline**

```
START → validate → generate (LCEL) → format → END
```

Shows the pattern discussed in Part 4 in a **multi-step** context:
- `validate_node` → plain Python (check input)
- `generate_node` → wraps an LCEL chain (the "worker")
- `format_node` → plain Python (post-process)

This is closer to how production graphs are built.

In [ ]:
# --- Scenario B: validate → generate (LCEL) → format ---
# Shows LCEL inside ONE node of a multi-node graph (no API key — RunnableLambda stand-in for llm chain)

class PipelineState(TypedDict):
    name: str
    valid: bool
    draft: str
    final: str


def validate_node(state: PipelineState) -> PipelineState:
    # Step 1: plain Python guard — reject empty names
    return {"valid": len(state["name"].strip()) > 0}


def route_after_validate(state: PipelineState) -> str:
    if state["valid"]:
        return "generate"
    else:
        return END


# Step 2: LCEL chain built FIRST, then called inside the node (same pattern as Part 4)
generate_chain = RunnableLambda(
    lambda x: f"Hello, {x['name']}! Welcome to LangGraph."
)


def generate_node(state: PipelineState) -> PipelineState:
    # Node wraps the LCEL chain — graph registers this function, not the chain itself
    draft = generate_chain.invoke({"name": state["name"]})
    return {"draft": draft}


def format_node(state: PipelineState) -> PipelineState:
    # Step 3: plain Python post-processing
    return {"final": state["draft"].upper()}


pipeline_graph = StateGraph(PipelineState)
pipeline_graph.add_node("validate", validate_node)
pipeline_graph.add_node("generate", generate_node)
pipeline_graph.add_node("format", format_node)

pipeline_graph.add_edge(START, "validate")
pipeline_graph.add_conditional_edges("validate", route_after_validate)
pipeline_graph.add_edge("generate", "format")
pipeline_graph.add_edge("format", END)

pipeline_app = pipeline_graph.compile()

# Valid input: runs all three nodes
good = pipeline_app.invoke({"name": "LangGraph", "valid": False, "draft": "", "final": ""})
print("Valid input:  ", good)

# Invalid input: validate → END (generate and format are skipped)
bad = pipeline_app.invoke({"name": "  ", "valid": False, "draft": "", "final": ""})
print("Invalid input:", bad)

### Key Takeaways

| Question | Answer |
|---|---|
| Do I need LCEL before LangGraph? | Learn LCEL first; many nodes wrap LCEL chains |
| Build LCEL, then put it in a node? | **Yes** — common pattern: `chain = prompt \| llm \| parser`, then call `chain.invoke()` inside `def my_node(state)` |
| Register the chain or the function? | Register the **node function**: `graph.add_node("llm", llm_node)` |
| Does every node need LCEL? | **No** — routing, validation, formatting can be plain Python |
| When is LangGraph worth it? | Multiple nodes, conditional routing, loops, tools, checkpointing, human-in-the-loop |
| When is LCEL enough? | Single linear step with no branching or persistence |

**Mental model:**
- **LCEL** = how one desk worker does their job
- **LangGraph** = the office floor plan deciding which desk runs next

## Exit Criteria Checklist

- [ ] LangGraph imports successfully
- [ ] You ran `START → greet → END` and got a transformed message
- [ ] You understand Part 4 reuses the LCEL chain **on purpose** for comparison (single-node graph = LCEL is enough)
- [ ] You can explain: LCEL = one step, LangGraph = orchestration across steps
- [ ] You know you register the **node function**, not the LCEL chain
- [ ] Scenario A: you saw conditional routing send tickets to different nodes
- [ ] Scenario B: you saw LCEL used inside one node of a multi-step pipeline
- [ ] You can explain why a graph is better than a `while` loop for agent control

**Next:** [1.1 First StateGraph](../../../Learning/LangGraph/01_beginner/1.1_first_stategraph.md) → `../01_beginner/1.1_first_stategraph.ipynb`